# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")

print("Dataset published on:", metadata['datePublished'])
print("Dataset identifier:", metadata['identifier'])
print("Dataset version:", metadata['version'])
print("Dataset keywords:", ', '.join(metadata['keywords']))


## 2. Data Overview

Review available record sets, fields, and their IDs from the dataset.

Note: All references are made by `@id` as per the FAIR² schema convention.

In [ ]:
# Retrieve record sets from metadata
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f" - RecordSet @id: {rs['@id']} | Name: {rs.get('name', '[no name]')}")

# List the fields (columns) for each record set
for rs in record_sets:
    print(f"\nFields for RecordSet {rs['@id']}:")
    fields = rs.get('fields', [])
    for field in fields:
        print(f"   Field @id: {field['@id']} | Name: {field.get('name', '[no name]')} | DataType: {field.get('dataType', '[unknown]')}")

# If the schema does not contain record sets, attempt to list distributions
if not record_sets:
    print("No recordSets found. Listing available distributions:")
    for distribution in metadata['distribution']:
        print(f" - Distribution @id: {distribution['@id']}")


## 3. Data Extraction

Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# If record sets are present, use their @id; otherwise, use distribution @id
if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    record_set_ids = [d['@id'] for d in metadata['distribution']]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet {record_set_id} with {len(df)} records.")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

# For demonstration, pick the first available record set
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"\nSelected RecordSet for further analysis: {selected_record_set_id}")
else:
    selected_record_set_id = None


## 4. Exploratory Data Analysis (EDA)

Explore and process the data for deeper insights. Common steps include filtering, normalization, and aggregation.

Reference fields using their `@id`.

In [ ]:
# Example: Filter records, normalize numeric fields, and group data

if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    # Identify numeric fields using fields metadata
    numeric_field_id = None
    group_field_id = None
    for rs in record_sets:
        if rs['@id'] == selected_record_set_id:
            for field in rs.get('fields', []):
                if field.get('dataType') in ['schema:Float', 'schema:Integer'] and field['@id'] in df.columns:
                    numeric_field_id = field['@id']
                    break
            for field in rs.get('fields', []):
                if field.get('dataType') in ['schema:Text'] and field['@id'] in df.columns:
                    group_field_id = field['@id']
                    break
    if not numeric_field_id:
        # fallback if metadata not present
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    print(f"Numeric field selected: {numeric_field_id}")

    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a text/categorical field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group_field if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated loading, exploring, and visualizing the FAIR² dataset defined by a Croissant schema using the `mlcroissant` library. Key findings included an overview of the available record sets and fields, basic statistical analysis, normalization, and visualization of selected data.

Refer to the dataset's metadata for detailed context on clinical and molecular variables. For further analysis, consult the full field descriptions and apply domain-specific filters as needed.